In [6]:
import bz2
import json
import pandas as pd

wikidata_json_file_path = "/home/yanghe/Downloads/latest-all.json.bz2"
# wikidata_lexemes_file_path = "/home/yanghe/Downloads/latest-lexemes.json.bz2"

In [2]:
# 统计json文件中有多少行
# line_count = 0
# with bz2.open(wikidata_json_file_path, 'rt') as file:
#     for _ in file:
#         line_count += 1 
#     print(line_count)

# print(line_count)

In [7]:
def get_multilingual_labels(data:dict) -> dict:
    """
    获取多语言标签
    """
    labels = {}
    label_keys = ["zh", "zh-hans", "zh-cn", "zh-tw", "yue", "kk", "kk-kz", "en"]
    
    for key in label_keys:
        value = data.get("labels", {}).get(key, {}).get("value")
        if value:
            labels[key] = value
    
    return labels


def get_multilingual_descriptions(data:dict) -> dict:
    """
    获取不同语言的描述
    """
    descriptions = {}
    description_keys = ["zh", "zh-hans", "zh-cn", "zh-tw", "yue", "kk", "kk-kz", "en"]
    
    for key in description_keys:
        value = data.get("descriptions", {}).get(key, {}).get("value")
        if value:
            descriptions[key] = str(value)
    
    return descriptions

def get_multilingual_aliases(data:dict) -> dict:
    """
    获取多语言别名
    """
    aliases = {}
    alias_keys = ["zh", "kk", "en"]
    
    for key in alias_keys:
        values = data.get("aliases", {}).get(key, [])
        if values:
            alias_list = [d.get("value") for d in values if d.get("value")]
            aliases[key] = ", ".join(alias_list)
    
    return aliases


def get_multilingual_sitelinks(data:dict) -> dict:
    """
    获取对应的wikipedia连接
    """
    sitelinks_prefix = {
        "zhwiki": "https://zh.wikipedia.org/wiki/",
        "zh_yuewiki": "https://zh-yue.wikipedia.org/wiki/",
        "kkwiki": "https://kk.wikipedia.org/wiki/",
        "enwiki": "https://en.wikipedia.org/wiki/"
    }

    sitelinks = {}
    for language_code, url_prefix in sitelinks_prefix.items():
        title = data.get("sitelinks", {}).get(language_code, {}).get("title")
        if title:
            sitelinks[language_code] = url_prefix + title

    return sitelinks

def modify_keys(dictionary:dict, prefix:str) -> dict:
    """
    调整字典的键，作为df数据的列名
    """
    modified_dict = {}
    for key, value in dictionary.items():
        modified_dict[prefix + '-' + key] = value
    return modified_dict


def merge_dicts(dict1, dict2):
    """
    合并两个字典,如果存在相同的键,则引发KeyError异常
    """
    # 先检查是否有重复键
    dict1_keys = set(dict1.keys())
    dict2_keys = set(dict2.keys())
    duplicate_keys = dict1_keys.intersection(dict2_keys)
    
    if duplicate_keys:
        raise KeyError(f"字典存在重复键: {', '.join(duplicate_keys)}")
    
    # 如果没有重复键,再进行合并
    merged = {k: v for d in (dict1, dict2) for k, v in d.items()}
    return merged

In [8]:
# 构建一个空字典 存储相关数据
item_info= dict()

with bz2.open(wikidata_json_file_path, 'rt') as wikidata_json_file, open('item_info.jsonl', 'w', encoding='utf-8') as item_info_jsonl_file:
    i = 0
    for line in wikidata_json_file:
        if i == 1000:
            break

        if len(line.strip()) < 100:
            continue
        
        # 文件中的数据类似 [{},{},{}] (可以说是一个列表，里面的元素是字典)
        if line.strip().endswith(","):
            json_data = line.strip()[:-1]

        # 解析json数据
        # json.loads() 是将 JSON 字符串解析为 Python 字典或列表对象的方法。它的参数是一个 JSON 字符串，返回一个 Python 对象。
        # json.load() 是从文件对象中读取 JSON 数据并解析为 Python 对象的方法。它的参数是一个文件对象，返回一个 Python 对象。
        data = json.loads(json_data)

        # 获取id
        id_value = data.get("id")
        item_info["id"] = id_value

        # 获取类型
        type_value = data.get("type")
        item_info["type"] = type_value

        # 获取多语言标签
        labels = get_multilingual_labels(data)
        item_info = merge_dicts(item_info, modify_keys(labels, "label"))

        # 获取不同语言的描述
        descriptions = get_multilingual_descriptions(data)
        item_info = merge_dicts(item_info, modify_keys(descriptions, "description"))

        # 获取别名
        aliases = get_multilingual_aliases(data)
        item_info = merge_dicts(item_info, modify_keys(aliases, "alias"))

        # 获取wikipedia链接
        sitelinks = get_multilingual_sitelinks(data)
        item_info = merge_dicts(item_info, sitelinks)
        # print(item_info)

        if i == 0:
            df = pd.DataFrame([item_info])
        else:
            df = pd.concat([df, pd.DataFrame([item_info])], ignore_index=True)

        json.dump(item_info, item_info_jsonl_file, ensure_ascii=False)
        item_info_jsonl_file.write('\n')
        
        # 将字典清空
        item_info.clear()

        i += 1


In [5]:
# df.to_csv('item_info.csv', index=False)
